# Creates VirtualZarr store from CESM2-WACCM-SSP245 NetCDF, then rechunks and writes to Icechunk store on s3.

- run on an m8g.4xlarge w/ 32 workers


In [44]:
import xarray as xr
import zarr
from obstore.store import from_url
import obstore as obs
from virtualizarr import open_virtual_mfdataset
from virtualizarr.parsers import HDFParser
from virtualizarr.registry import ObjectStoreRegistry
from distributed import Client
import icechunk
from icechunk.xarray import to_icechunk
import warnings

warnings.filterwarnings("ignore", module="zarr.*")
warnings.filterwarnings("ignore", module="numcodecs.*")

zarr.config.set({"async.concurrency": 128})

print(zarr.config.get("async.concurrency"))

import warnings

warnings.filterwarnings("ignore")

128


In [2]:
client = Client(n_workers=32)
client

2025-09-18 19:35:19,704 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='cluster-kldcb.dask.host', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/opt/coiled/env/lib/python3.13/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/opt/coiled/env/lib/python3.13/site-packages/tornado/web.py", line 3375, in wrapper
    return method(self, *args, **kwargs)
  File "/opt/coiled/env/lib/python3.13/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-token-expiration if necessary


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://cluster-kldcb.dask.host/jupyter/proxy/8787/status,
Dashboard: https://cluster-kldcb.dask.host/jupyter/proxy/8787/status,Workers: 32
Total threads: 32,Total memory: 60.69 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45631,Workers: 0
Dashboard: https://cluster-kldcb.dask.host/jupyter/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34383,Total threads: 1
Dashboard: https://cluster-kldcb.dask.host/jupyter/proxy/33661/status,Memory: 1.90 GiB
Nanny: tcp://127.0.0.1:36421,


2025-09-18 19:35:27,944 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='cluster-kldcb.dask.host', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/opt/coiled/env/lib/python3.13/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/opt/coiled/env/lib/python3.13/site-packages/tornado/web.py", line 3375, in wrapper
    return method(self, *args, **kwargs)
  File "/opt/coiled/env/lib/python3.13/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-token-expiration if necessary


In [7]:
bucket = "s3://carbonplan-srm/"
prefix = "input/tensor/CESM2-WACCM-SSP245/netcdf"
virtual_ic_prefix = "input/tensor/CESM2-WACCM-SSP245/icechunk/virtual_icechunk"
ic_prefix = "input/tensor/CESM2-WACCM-SSP245/icechunk/icechunk"
store = from_url(bucket, region="us-west-2")
registry = ObjectStoreRegistry({bucket: store})
drop_variables = [
    "gw",
    "hyam",
    "hybm",
    "P0",
    "hyai",
    "hybi",
    "ndbase",
    "nsbase",
    "nbdate",
    "nbsec",
    "mdt",
    "date",
    "datesec",
    "time_bnds",
    "date_written",
    "time_written",
    "ndcur",
    "nscur",
    "co2vmr",
    "ch4vmr",
    "n2ovmr",
    "f11vmr",
    "f12vmr",
    "sol_tsi",
    "nsteph",
]
parser = HDFParser(drop_variables=drop_variables)

In [8]:
stream = obs.list_with_delimiter(store, prefix=prefix, return_arrow=True)
netcdf_list = list(stream["objects"]["path"].to_numpy())
netcdf_list.remove(prefix)
netcdf_urls = [bucket + netcdf_path for netcdf_path in netcdf_list]

# SPECIFIC EXCLUSION OF SOME ENSEMBLE MEMBERS FOR CESM-WACCM-G6-1.5K ONLY!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
exclude_numbers = ["001", "002", "003", "004", "005"]
netcdf_urls = [
    path
    for path in netcdf_urls
    if not any(f"CMIP6-SSP2-4.5-WACCM.{num}." in path for num in exclude_numbers)
]
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [20]:
ensemble_members = ["006", "007", "008", "009", "010"]
grouped = {}

for member in ensemble_members:
    grouped[member] = [path for path in netcdf_urls if f".{member}." in path]

In [38]:
def preprocess(ds):
    """
    get ensemble member from ds attrs filename
    """
    ensemble = ds.attrs["case"].rsplit(".")[-1]

    ds = ds.expand_dims({"ensemble_member": [ensemble]})
    return ds

In [ ]:
# ds = xr.open_dataset('https://carbonplan-srm.s3.us-west-2.amazonaws.com/input/tensor/CESM-WACCM-G6-1.5K/netcdf/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.001.cam.h1.FLDS.20350101-20441231.nc',engine='h5netcdf')
# ensemble = ds.attrs['initial_file'].rsplit('.')[-5]

In [ ]:
# vds = open_virtual_dataset(netcdf_urls[0],registry=registry,
#     parser=parser)

In [ ]:
# netcdf_urls
# combined_vds = open_virtual_mfdataset(
#     netcdf_urls,
#     registry=registry,
#     parser=parser,
#     combine="by_coords",
#     combine_attrs="drop_conflicts",
#     loadable_variables=["lat", "lev", "ilev", "time", "nbnd", "lon"],
#     parallel="dask",
# )

In [42]:
list(grouped.keys())[0:2]

['006', '007']

In [51]:
eds_list = []
for ensemble in list(grouped.keys()):
    combined_vds = open_virtual_mfdataset(
        grouped[ensemble],
        registry=registry,
        parser=parser,
        preprocess=preprocess,
        combine="by_coords",
        combine_attrs="drop_conflicts",
        loadable_variables=["lat", "lev", "ilev", "time", "nbnd", "lon"],
        parallel="dask",
    )
    eds_list.append(combined_vds)

/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:

In [53]:
for ds in eds_list:
    print(ds)

<xarray.Dataset> Size: 44GB
Dimensions:          (ensemble_member: 1, time: 20075, lat: 192, lon: 288,
                      lev: 70, ilev: 71)
Coordinates:
  * ensemble_member  (ensemble_member) object 8B '006'
  * lat              (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon              (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
  * lev              (lev) float64 560B 5.96e-06 9.827e-06 ... 976.3 992.6
  * ilev             (ilev) float64 568B 4.5e-06 7.42e-06 ... 985.1 1e+03
  * time             (time) object 161kB 2015-01-01 00:00:00 ... 2069-12-31 0...
Data variables:
    FLDS             (ensemble_member, time, lat, lon) float32 4GB ManifestAr...
    FSDS             (ensemble_member, time, lat, lon) float32 4GB ManifestAr...
    PRECT            (ensemble_member, time, lat, lon) float32 4GB ManifestAr...
    PS               (ensemble_member, time, lat, lon) float32 4GB ManifestAr...
    QREFHT           (ensemble_member, time, lat, lon) floa

In [ ]:
combined_vds

In [ ]:
combined_vds = combined_vds.drop_vars(["ilev", "lev"])

In [ ]:
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        "s3://carbonplan-srm/",
        store=icechunk.s3_store(region="us-west-2"),
    ),
)


storage = icechunk.s3_storage(
    bucket="carbonplan-srm", prefix=virtual_ic_prefix, from_env=True
)
repo = icechunk.Repository.open_or_create(storage, config)
session = repo.writable_session("main")

In [ ]:
combined_vds.vz.to_icechunk(session.store)
snapshot_id = session.commit("virtual_CESM2-WACCM-SSP245")
print(snapshot_id)
repo.save_config()

# read

In [ ]:
credentials = icechunk.containers_credentials(
    {
        "s3://carbonplan-srm": icechunk.s3_credentials(),
    }
)

vz_repo = icechunk.Repository.open(
    storage=storage,
    config=config,
    authorize_virtual_chunk_access=credentials,
)
vz_session = vz_repo.readonly_session("main")

In [ ]:
ds = xr.open_zarr(
    vz_session.store,
    zarr_format=3,
    consolidated=False,
    chunks={},
)

ds

In [ ]:
ds.isel(time=0).PS.plot()

In [ ]:
write_storage_config = icechunk.s3_storage(bucket="carbonplan-srm", prefix=ic_prefix)
write_repo = icechunk.Repository.open_or_create(write_storage_config)
write_session = write_repo.writable_session("main")

In [ ]:
# ds.chunk({'time':8000, 'lat':48,'lon':72}) # split lat lon chunking by factor of 4. ~100MB chunks

In [ ]:
ds = ds.drop_encoding()

In [ ]:
rds = ds.chunk({"time": 8000, "lat": 48, "lon": 72})

In [ ]:
to_icechunk(rds, write_session)

In [ ]:
first_snapshot = write_session.commit("create ic store")

In [ ]:
first_snapshot

In [ ]:
rtds = xr.open_zarr(write_session.store)
rtds